In [ ]:
import coiled

import fsspec
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import dask
import sparse
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [ ]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [ ]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=2,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [ ]:
import re

def parse_metadata_from_uri(uri: str):
    """
    Extracts year, chunk, and variable from an S3 URI string.
    """
    uri = uri.values.tolist()
    # This regex captures year, chunk, and variable in your filename format
    pattern = r'(\d{4}_\d{4}).*?__(\d+_-?\d+_\d+_-?\d+)__([a-zA-Z0-9_]+)(?:_pixel_yr)?_\1'
    match = re.search(pattern, uri)
    
    if match:
        interval = match.group(1)
        chunk_id = match.group(2)
        variable = match.group(3)
    else:
        interval, chunk_id, variable = None, None, None
    
    return interval, chunk_id, variable

In [ ]:
gross_emis_CO2_only_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/2016_2017/_pixel_yr/4000_pixels/20250507/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_2015_2016.tif"], name = 'emis_all_C_pools_CO2_only')
gross_emis_all_gases_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/2016_2017/_pixel_yr/4000_pixels/20250507/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_2015_2016.tif"], name = 'emis_all_C_pools_all_gases')
node_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_1/land_state_node/standard_model/annual_intervals/2015_2016/4000_pixels/20250507/00N_020E__23_-4_24_-3__land_state_node_2015_2016.tif"], name = 'node_codes')
print(gross_emis_CO2_only_tile_uri)
print(node_tile_uri)
print(type(emis_tile_uri))

In [ ]:
interval, chunk_id, variable = parse_metadata_from_uri(gross_emis_CO2_only_tile_uri)
print(interval)
print(chunk_id)
print(variable)

In [ ]:
model_version = "version_0_3_2"
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
run_date = "20250507"
interval = "2015_2016"

In [ ]:
gross_emis_CO2_2016_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif"], 
                                         name = 'emis_all_C_pools_CO2_only')
gross_emis_all_gases_2016_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif"], 
                                         name = 'emis_all_C_pools_all_gases')
node_2016_tile_uri = pd.Series([f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__land_state_node_{interval}.tif"], 
                          name = 'node_codes')
print(gross_emis_CO2_2016_tile_uri)
print(gross_emis_all_gases_2016_tile_uri)
print(node_2016_tile_uri)
print(type(gross_emis_CO2_only_2016_tile_uri))

In [ ]:
def make_xarray_chunks(tile_uris):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True
        # chunks={'x': 1000, 'y':1000}
    ).squeeze().persist()

    return xarray_chunks

In [ ]:
layer_xarray_chunks = make_xarray_chunks(gross_emis_CO2_only_2016_tile_uri)
node_xarray_chunks = make_xarray_chunks(node_2016_tile_uri)
print(layer_xarray_chunks)
node_xarray_chunks

In [ ]:
gross_emis_CO2_2016 = xr.open_mfdataset(
    gross_emis_CO2_2016_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

gross_emis_all_gases_2016 = xr.open_mfdataset(
    gross_emis_all_gases_2016_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

nodes_2016 = xr.open_mfdataset(
    node_2016_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

print(gross_emis_CO2_2016)
print(gross_emis_all_gases_2016)
nodes_2016

In [ ]:
gross_emis_CO2_sub, nodes_aligned = xr.align(gross_emis_CO2, nodes, join="inner")
gross_emis_all_gases_sub, nodes_aligned = xr.align(gross_emis_all_gases, nodes, join="inner")
print(gross_emis_CO2_sub)
print(gross_emis_all_gases_sub)
nodes_aligned

In [ ]:
node_data = nodes_aligned.band_data
node_data.name = 'state_node'
node_data

In [ ]:
node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
                       2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
                       2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
                      dtype=np.uint32)

In [ ]:
gross_emis_CO2__by_node = xarray_reduce(
    gross_emis_CO2_sub.band_data,
    node_data,
    func='sum',
    expected_groups=(node_codes),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
    
)

gross_emis_all_gases__by_node = xarray_reduce(
    gross_emis_all_gases_sub.band_data,
    node_data,
    func='sum',
    expected_groups=(node_codes),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
    
)

print(gross_emis_CO2__by_node)
print(gross_emis_all_gases__by_node)

In [ ]:
gross_emis_CO2_result = gross_emis_CO2__by_node.compute()
gross_emis_all_gases_result = gross_emis_all_gases__by_node.compute()

In [ ]:
print(gross_emis_CO2_result)
print(gross_emis_all_gases_result)

In [ ]:
gross_emis_CO2_sparse_data = gross_emis_CO2_result.data
gross_emis_all_gases_sparse_data = gross_emis_all_gases_result.data

# Step 3: Extract coordinates and values
dim_names = gross_emis_CO2_result.dims
indices = gross_emis_CO2_sparse_data.coords
gross_emis_CO2_values = gross_emis_CO2_sparse_data.data
gross_emis_all_gases_values = gross_emis_all_gases_sparse_data.data

# Step 4: Map dimension indices to coordinate values
gross_emis_CO2_coord_dict = {
    dim: gross_emis_CO2_result.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
gross_emis_CO2_coord_dict["value"] = gross_emis_CO2_values

gross_emis_all_gases_coord_dict = {
    dim: gross_emis_all_gases_result.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
gross_emis_all_gases_coord_dict["value"] = gross_emis_all_gases_values

gross_emis_CO2_df = pd.DataFrame(gross_emis_CO2_coord_dict)
gross_emis_CO2_df["year"] = 2016
gross_emis_CO2_df["variable"] = "gross_emissions__all_C_pools__CO2_only__MgCO2"

gross_emis_all_gases_df = pd.DataFrame(gross_emis_all_gases_coord_dict)
gross_emis_all_gases_df["year"] = 2016
gross_emis_all_gases_df["variable"] = "gross_emissions__all_C_pools__all_gases__MgCO2"

In [ ]:
pd.concat([gross_emis_CO2_df, gross_emis_all_gases_df])

In [ ]:
df.head()

In [ ]:
df[(df.state_node == 2212210)]